# 01.3 — Manage, monitor, and secure AI systems: lab

Four things, in order: quota and rate limits, monitoring, retrieval health,
and security.

**Cost:** under $1. Section 2 deliberately triggers throttling with a burst of
tiny calls — a few thousand tokens. Nothing hourly is created.

**Permissions:** sections 1–4 need read access. Section 5 (RBAC) is read-only
unless you have **User Access Administrator**; the write path is shown but
guarded by a flag you must set deliberately.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "scripts"))
from ai103 import cfg, credential, chat_client, project_client, embed

RG = cfg["AZURE_RESOURCE_GROUP"]
ACCOUNT = cfg["AZURE_AI_FOUNDRY_RESOURCE"]
SUB = cfg["AZURE_SUBSCRIPTION_ID"]
LOCATION = cfg["AZURE_LOCATION"]

# Set to True only if you hold User Access Administrator and want section 5 to write.
ALLOW_ROLE_ASSIGNMENT = False

print(f"{ACCOUNT} in {LOCATION}")

## 1. Quota headroom

Quota is a **subscription + region + model family** ceiling measured in TPM.
Every deployment draws from it. The first thing to know about your environment
is how much slack you have, because that number decides whether you can scale at
all.

In [ ]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

arm = CognitiveServicesManagementClient(credential(), SUB)

rows = []
for u in arm.usages.list(location=LOCATION):
    limit, used = (u.limit or 0), (u.current_value or 0)
    if limit > 0:
        rows.append((u.name.value if u.name else "?", used, limit))

rows.sort(key=lambda r: -r[1])
print(f"{'quota':<50}{'used':>9}{'limit':>9}{'free':>9}{'%':>7}")
print("-" * 84)
for name, used, limit in rows[:15]:
    print(f"{name[:48]:<50}{used:>9.0f}{limit:>9.0f}{limit - used:>9.0f}{used / limit:>7.0%}")

print("\nDeployments drawing on it:")
for d in arm.deployments.list(RG, ACCOUNT):
    print(f"  {d.name:<30}{d.sku.name:<20}capacity={d.sku.capacity} ({d.sku.capacity * 1000:,} TPM)")

## 2. Rate limits are a different thing entirely

Quota failures happen at **deployment creation** and return 400. Rate limits
happen at **request time** and return 429 with a `Retry-After` header.

Azure derives RPM from TPM at roughly **6 RPM per 1,000 TPM**. A 30K TPM
deployment gives about 180 requests per minute *however small those requests are*
— which is why a chatty workload with tiny prompts hits the RPM wall while using
a fraction of its token budget.

The burst below is intentionally aggressive. If your deployment has generous
capacity you may see no 429s at all; that is a valid result and the handler is
still the code you would ship.

In [ ]:
import time, random
from concurrent.futures import ThreadPoolExecutor

client = chat_client()
DEPLOYMENT = cfg["MODEL_MINI"]


def tiny_call(i: int):
    try:
        r = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=[{"role": "user", "content": f"Reply with the number {i}."}],
            max_tokens=5,
            temperature=0.0,
        )
        return ("ok", r.usage.total_tokens, None)
    except Exception as exc:  # noqa: BLE001
        status = getattr(exc, "status_code", None)
        retry_after = None
        headers = getattr(getattr(exc, "response", None), "headers", None)
        if headers:
            retry_after = headers.get("retry-after")
        return (str(status), 0, retry_after)


with ThreadPoolExecutor(max_workers=30) as pool:
    outcomes = list(pool.map(tiny_call, range(60)))

ok = sum(1 for o in outcomes if o[0] == "ok")
throttled = [o for o in outcomes if o[0] == "429"]
print(f"succeeded : {ok}/60")
print(f"throttled : {len(throttled)}")
print(f"tokens    : {sum(o[1] for o in outcomes):,} — nowhere near the TPM limit")
if throttled:
    print(f"Retry-After values seen: {sorted({t[2] for t in throttled if t[2]})}")
else:
    print("no throttling — your deployment has headroom for this burst")

### The retry you should actually ship

Two rules. **Honour `Retry-After`** — the service knows when capacity frees up
better than your backoff curve does. And **add jitter**, because synchronised
clients retrying on identical exponential schedules recreate the burst that caused
the throttle.

In [ ]:
def call_with_backoff(prompt: str, *, max_attempts: int = 5):
    for attempt in range(max_attempts):
        try:
            return client.chat.completions.create(
                model=DEPLOYMENT,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=30,
            )
        except Exception as exc:  # noqa: BLE001
            if getattr(exc, "status_code", None) != 429 or attempt == max_attempts - 1:
                raise
            headers = getattr(getattr(exc, "response", None), "headers", {}) or {}
            server_hint = headers.get("retry-after")
            wait = float(server_hint) if server_hint else 2**attempt
            wait += random.uniform(0, 0.5 * wait)  # jitter: break the thundering herd
            print(f"  429 — sleeping {wait:.1f}s (attempt {attempt + 1})")
            time.sleep(wait)
    raise RuntimeError("unreachable")


print(call_with_backoff("Name one way to reduce 429 errors. One sentence.").choices[0].message.content)

## 3. Monitoring

Two sources, and the exam distinguishes them:

- **Metrics** — pre-aggregated numeric series, available immediately, queried via
  Azure Monitor.
- **Logs** — per-request records, require a diagnostic setting, land in
  `AzureDiagnostics`, and take up to two hours to appear.

Start with metrics because they need no setup.

In [ ]:
import datetime as dt
from azure.monitor.query import MetricsQueryClient, MetricAggregationType

metrics = MetricsQueryClient(credential())
account = arm.accounts.get(RG, ACCOUNT)

try:
    result = metrics.query_resource(
        account.id,
        metric_names=["ProcessedPromptTokens", "GeneratedTokens", "AzureOpenAIRequests"],
        timespan=dt.timedelta(hours=2),
        granularity=dt.timedelta(minutes=15),
        aggregations=[MetricAggregationType.TOTAL],
    )
    for metric in result.metrics:
        total = sum(
            p.total or 0 for series in metric.timeseries for p in series.data
        )
        print(f"{metric.name:<26} last 2h total: {total:,.0f}")
except Exception as exc:  # noqa: BLE001
    print(f"metrics unavailable — {type(exc).__name__}: {exc}")
    print("Needs 'Monitoring Reader' (or Reader) on the resource.")

### KQL against Log Analytics

Everything from a Cognitive Services account lands in the generic
**`AzureDiagnostics`** table — there is no dedicated `AzureOpenAI…` table, and
assuming there is is a common exam trap.

If this returns nothing: the diagnostic setting is missing (README section A), or
you enabled it less than two hours ago.

In [ ]:
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

logs = LogsQueryClient(credential())
workspace_id = cfg.get("AZURE_LOG_ANALYTICS_WORKSPACE_ID") or cfg.get("AZURE_LOG_ANALYTICS_WORKSPACE")

QUERY = """
AzureDiagnostics
| where ResourceProvider == "MICROSOFT.COGNITIVESERVICES"
| where TimeGenerated > ago(24h)
| summarize calls      = count(),
            throttled  = countif(resultSignature_d == 429),
            failed     = countif(resultSignature_d >= 500),
            p50_ms     = percentile(DurationMs, 50),
            p95_ms     = percentile(DurationMs, 95)
  by OperationName
| order by calls desc
"""

if not workspace_id:
    print("AZURE_LOG_ANALYTICS_WORKSPACE_ID not set — showing the query only.")
    print(QUERY)
else:
    try:
        r = logs.query_workspace(workspace_id, QUERY, timespan=dt.timedelta(days=1))
        if r.status == LogsQueryStatus.SUCCESS and r.tables and r.tables[0].rows:
            table = r.tables[0]
            print(" | ".join(table.columns))
            for row in table.rows:
                print(" | ".join(str(v) for v in row))
        else:
            print("no rows — diagnostic settings missing, or enabled under 2h ago")
    except Exception as exc:  # noqa: BLE001
        print(f"{type(exc).__name__}: {exc}")

## 4. Safety events, grounding, and drift — the three model signals

### Safety events

The content filter reports itself in two places on every response:
`prompt_filter_results` (what it thought of your input) and
`finish_reason == "content_filter"` (it stopped the output). A blocked *prompt*
raises a 400 with code `content_filter` instead.

This is the signal you log. Unit 01.4 configures the filter itself.

In [ ]:
def probe_filter(prompt: str):
    try:
        r = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=40,
        )
        choice = r.choices[0]
        pfr = getattr(r, "prompt_filter_results", None)
        return {
            "blocked": choice.finish_reason == "content_filter",
            "finish_reason": choice.finish_reason,
            "prompt_annotations": bool(pfr),
        }
    except Exception as exc:  # noqa: BLE001
        code = getattr(exc, "code", None) or getattr(exc, "status_code", None)
        return {"blocked": True, "finish_reason": f"request rejected ({code})", "prompt_annotations": True}


for p in [
    "Summarise the benefits of hybrid search in one sentence.",
    "Explain how the content filter severity levels work.",
]:
    print(f"{p[:52]:<55} {probe_filter(p)}")

print("\nLog finish_reason on every production call. A rising rate of")
print("'content_filter' is either an attack, or a filter that is too strict.")

### Grounding quality

"Grounding quality" means: did the answer stay inside the retrieved context, or
did the model invent the difference? The cell below is a deliberately crude
detector — it asks a model to judge. Unit 01.4 replaces it with
`GroundednessEvaluator`, which is the same idea with a calibrated rubric and a
reason field.

The point here is what you *do* with the number: sample a percentage of production
traffic, score it, and alert when the rolling average drops.

In [ ]:
CONTEXT = (
    "The Contoso XR-200 has a 24-month warranty. Consumable filters are excluded. "
    "Claims require the original receipt."
)

CASES = [
    ("How long is the warranty?", "24 months."),
    ("How long is the warranty?", "24 months, extendable to 36 for a fee."),  # invented
    ("Are filters covered?", "No, consumable filters are excluded."),
]

JUDGE = (
    "Score 1-5 how fully the RESPONSE is supported by the CONTEXT. "
    "5 = every claim supported. 1 = mostly invented. Reply with only the digit."
)

for question, answer in CASES:
    verdict = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": JUDGE},
            {"role": "user", "content": f"CONTEXT:\n{CONTEXT}\n\nQUESTION: {question}\nRESPONSE: {answer}"},
        ],
        temperature=0.0,
        max_tokens=3,
    )
    print(f"score {verdict.choices[0].message.content.strip()}  <-  {answer}")

### Data drift

Drift is measured, not felt. Embed a baseline sample of production inputs, keep
the centroid, then measure how far new traffic sits from it. A rising mean
distance means users are asking about things your index and prompts were never
designed for — usually the earliest warning that relevance is about to fall off.

In [ ]:
BASELINE = [
    "How do I reset the XR-200?",
    "What is covered by the warranty?",
    "My unit will not power on.",
    "How do I replace the filter?",
    "Where do I find the serial number?",
]

NEW_TRAFFIC = [
    "How do I reset the device?",                      # on-distribution
    "Can I integrate the XR-200 with Home Assistant?",  # new topic
    "What is your refund policy for EU customers?",     # new topic
]

base_vecs = embed(BASELINE)
centroid = [sum(col) / len(col) for col in zip(*base_vecs)]


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / ((sum(x * x for x in a) ** 0.5) * (sum(y * y for y in b) ** 0.5))


base_dists = [1 - cosine(v, centroid) for v in base_vecs]
threshold = max(base_dists)
print(f"baseline max distance from centroid: {threshold:.4f}\n")

for q, v in zip(NEW_TRAFFIC, embed(NEW_TRAFFIC)):
    d = 1 - cosine(v, centroid)
    flag = "DRIFT" if d > threshold else "ok   "
    print(f"{flag}  {d:.4f}  {q}")

> **Exam note.** This detects **data** drift — the inputs changed. It says nothing
> about **quality** drift, where identical inputs start producing worse answers
> after a model version upgrade. The only reliable detector for that is a
> **scheduled evaluation against a fixed golden dataset**, which unit 01.4 builds.

## 5. Search index and ingestion health

Skipped cleanly if you have no Azure AI Search service — it bills hourly and unit
00 leaves it optional.

The check that matters most is **document count reconciliation**: source documents
versus indexed documents. A gap means the indexer failed silently.

In [ ]:
if not cfg.get("AZURE_SEARCH_ENDPOINT"):
    print("AZURE_SEARCH_ENDPOINT not set — skipping. Unit 05.1 provisions Search.")
else:
    from ai103 import search_index_client

    try:
        six = search_index_client()
        stats = six.get_service_statistics()
        counters = stats["counters"]
        for key in ("documentCount", "indexesCount", "storageSize", "vectorIndexSize"):
            c = counters.get(key)
            if c:
                quota = c.get("quota")
                pct = f"  ({c['usage'] / quota:.1%} of quota)" if quota else ""
                print(f"{key:<18}{c['usage']:>14,}{pct}")

        print("\nPer-index:")
        for idx in six.list_index_names():
            s = six.get_index_statistics(idx)
            print(f"  {idx:<28}{s['document_count']:>10,} docs  {s['storage_size']:>14,} bytes")
    except Exception as exc:  # noqa: BLE001
        print(f"{type(exc).__name__}: {exc}")
        print("Needs 'Search Service Contributor' or 'Search Index Data Reader'.")

In [ ]:
# Indexer health. A run can report SUCCESS and still have dropped documents —
# that is the partially-failed-skillset failure mode from the README.
if cfg.get("AZURE_SEARCH_ENDPOINT"):
    try:
        from azure.search.documents.indexes import SearchIndexerClient

        sic = SearchIndexerClient(endpoint=cfg["AZURE_SEARCH_ENDPOINT"], credential=credential())
        names = sic.get_indexer_names()
        if not names:
            print("no indexers — a push-based pipeline needs its own telemetry instead")
        for name in names:
            st = sic.get_indexer_status(name)
            last = st.last_result
            print(f"{name}: status={st.status}")
            if last:
                print(
                    f"  last run {last.status}: "
                    f"{last.item_count} items, {last.failed_item_count} failed, "
                    f"{len(last.errors or [])} errors, {len(last.warnings or [])} warnings"
                )
                for w in (last.warnings or [])[:3]:
                    print(f"    WARN {w.key}: {w.message[:90]}")
    except Exception as exc:  # noqa: BLE001
        print(f"{type(exc).__name__}: {exc}")

## 6. Security posture

### What is actually configured right now

In [ ]:
p = account.properties
acls = getattr(p, "network_acls", None)

posture = {
    "managed identity": account.identity.type if account.identity else "NONE",
    "custom subdomain": p.custom_sub_domain_name or "NONE (Entra ID auth impossible)",
    "disableLocalAuth": getattr(p, "disable_local_auth", False),
    "publicNetworkAccess": p.public_network_access,
    "networkAcls default": getattr(acls, "default_action", "n/a"),
    "ip rules": len(getattr(acls, "ip_rules", []) or []),
    "private endpoints": len(getattr(p, "private_endpoint_connections", []) or []),
}
for k, v in posture.items():
    print(f"{k:<24}: {v}")

print("\nGaps against the production target:")
if not getattr(p, "disable_local_auth", False):
    print("  - API keys still work. Keyless is a client choice until disableLocalAuth=true.")
if p.public_network_access != "Disabled":
    print("  - Endpoint is reachable from the internet.")
if not getattr(p, "private_endpoint_connections", None):
    print("  - No private endpoint. Do NOT disable public access before creating one.")

### Keyless, demonstrated

`DefaultAzureCredential` is the *client* half. `disableLocalAuth` is the
*resource* half. Only the second one actually stops anyone using a key.

In [ ]:
token = credential().get_token("https://cognitiveservices.azure.com/.default")
expires = dt.datetime.fromtimestamp(token.expires_on, dt.timezone.utc)
print(f"Entra ID token acquired, expires {expires:%H:%M UTC} — nothing to rotate or leak\n")

try:
    keys = arm.accounts.list_keys(RG, ACCOUNT)
    print("Keys still exist and still work.")
    print(f"key1 starts with: {keys.key1[:6]}…")
    print("\nTo enforce keyless:")
    print(f"  az resource update --ids {account.id} --set properties.disableLocalAuth=true")
except Exception as exc:  # noqa: BLE001
    print(f"cannot read keys ({type(exc).__name__}) — you lack Cognitive Services Contributor")

### RBAC audit

The Foundry roles were **recently renamed** — *Azure AI User* → **Foundry User**,
*Azure AI Project Manager* → **Foundry Project Manager**, and so on. The role IDs
and permissions are unchanged, so either name may appear in your tenant and in
exam questions.

In [ ]:
from azure.mgmt.authorization import AuthorizationManagementClient

auth = AuthorizationManagementClient(credential(), SUB)
scope = account.id
names = {rd.id: rd.role_name for rd in auth.role_definitions.list(scope)}

print(f"Assignments at {ACCOUNT}:")
for ra in auth.role_assignments.list_for_scope(scope, filter="atScope()"):
    print(f"  {names.get(ra.role_definition_id, '(unknown)'):<44}{ra.principal_type}")

print("\nInherited from subscription/resource group (these can mask your intent):")
for ra in auth.role_assignments.list_for_scope(scope):
    role = names.get(ra.role_definition_id, "")
    if role in {"Owner", "Contributor", "Reader"} and ra.scope != scope:
        print(f"  {role:<44}at {ra.scope.split('/')[-1]}")

In [ ]:
ROLE_IDS = {
    "Foundry User (was Azure AI User)": "53ca6127-db72-4b80-b1b0-d745d6d5456d",
    "Foundry Project Manager": "eadc314b-1a2d-4efa-be10-5d325db5065e",
    "Foundry Account Owner": "e47c6f54-e4a2-4754-9501-8e0985b135e1",
    "Foundry Owner": "c883944f-8b7b-4483-af10-35834be79c4a",
    "Cognitive Services User": "a97b65f3-24c7-4388-baec-2e87135dc908",
    "Cognitive Services OpenAI User": "5e0bd9bd-7b93-4f28-af87-19fc36ad61bd",
    "Cognitive Services OpenAI Contributor": "a001fd3d-188f-4b5d-821b-7da978bf7442",
    "Search Index Data Reader": "1407120a-92aa-4202-b7e9-c0e197c71c8f",
    "Search Index Data Contributor": "8ebe5a00-799e-43f5-93ac-243d3dce84a7",
    "Search Service Contributor": "7ca78c08-252a-4471-8644-bb5ff32d4ba0",
}

print(f"{'role':<40}{'exists in tenant':<18}id")
print("-" * 90)
for label, rid in ROLE_IDS.items():
    try:
        rd = auth.role_definitions.get(f"/subscriptions/{SUB}", rid)
        print(f"{label:<40}{'yes — ' + rd.role_name:<18}{rid}")
    except Exception:  # noqa: BLE001
        print(f"{label:<40}{'not found':<18}{rid}")

In [ ]:
# Guarded write path. Assigns Foundry User to your own principal on the account.
# Needs User Access Administrator or Owner.
if not ALLOW_ROLE_ASSIGNMENT:
    print("ALLOW_ROLE_ASSIGNMENT is False — skipping the write.")
    print("Set it True in cell 1 if you hold User Access Administrator.")
else:
    import uuid
    import subprocess

    principal_id = subprocess.run(
        ["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
        capture_output=True, text=True, shell=True,
    ).stdout.strip()

    role_id = ROLE_IDS["Foundry User (was Azure AI User)"]
    try:
        auth.role_assignments.create(
            scope=scope,
            role_assignment_name=str(uuid.uuid4()),
            parameters={
                "role_definition_id": f"/subscriptions/{SUB}/providers/Microsoft.Authorization/roleDefinitions/{role_id}",
                "principal_id": principal_id,
                "principal_type": "User",
            },
        )
        print("assigned — allow up to 5 minutes for propagation")
    except Exception as exc:  # noqa: BLE001
        print(f"{type(exc).__name__}: {exc}")

## 7. Cleanup

This lab is almost entirely read-only. It created no deployments, agents, or
indexes. The only possible residue is a role assignment, if you enabled the guard.

The cell also reminds you what is billing by the hour right now — the thing worth
checking at the end of every session.

In [ ]:
from azure.mgmt.resource import ResourceManagementClient

res = ResourceManagementClient(credential(), SUB)

HOURLY = {
    "Microsoft.Search/searchServices": "Azure AI Search — billed hourly, delete when idle",
    "Microsoft.OperationalInsights/workspaces": "Log Analytics — billed per GB ingested",
}

print(f"Resources in {RG}:")
for r in res.resources.list_by_resource_group(RG):
    note = HOURLY.get(r.type, "")
    print(f"  {r.name:<40}{r.type:<48}{note}")

print("\nThis lab created: nothing billable.")
if ALLOW_ROLE_ASSIGNMENT:
    print("It may have created one role assignment — remove it in IAM if unwanted.")

## Exercise

Solutions at the bottom of [quiz.md](quiz.md).

1. **Separate the two failures.** Trigger a `429` (section 2) and an
   `InsufficientQuota` (try creating a deployment with capacity 100000). Record
   the HTTP status, error code, and which headers each carries. Write two
   sentences on why an application must handle them completely differently.

2. **Write a cost query.** Adapt the token-burn KQL from the README to report
   estimated USD per deployment per day, using the rates from unit 01.1. Which
   deployment is your largest cost? Is it the one you expected?

3. **Make drift detection stateful.** The detector in section 4 recomputes the
   baseline each run, so it can never fire in production. Persist the centroid and
   threshold to a file, then re-run with only new traffic. What else must you
   store, and how would you decide when to re-baseline?

4. **Build the least-privilege matrix.** For a RAG application, list the exact
   role assignments needed: which principal, which role, which scope. Cover the
   web app identity, the Foundry account identity, and the Search service
   identity. Then explain why the app identity should *not* get
   `Search Index Data Contributor`.

5. **Reason about locking the door.** Without running it, describe exactly what
   breaks if you set `publicNetworkAccess=Disabled` on your Foundry resource right
   now, and list every step required to make it work again.

In [ ]:
# Your work here.

## Next

[01.4 — Implement responsible AI across generative AI and agentic systems](../04_responsible_ai/README.md)